In [20]:
import os
os.environ["GOOGLE_API_KEY"] = "IzaSyBcqInDNJ3WzXNV8EkwSCTV9xkk"

In [21]:
!pip install -q youtube-transcript-api langchain-community \
               langchain-google-genai faiss-cpu tiktoken python-dotenv google-generativeai


In [32]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

**Indexing**

In [23]:
from youtube_transcript_api._errors import TranscriptsDisabled

video_id = "Gfr50f6ZBvo"

try:
    fetched_transcript = YouTubeTranscriptApi().fetch(video_id, languages=['en'])
    transcript_list = fetched_transcript.to_raw_data()
    transcript = " ".join(chunk["text"] for chunk in transcript_list)
    print(transcript)

except TranscriptsDisabled:
    print("No captions available for this video.")
except Exception as e:
    print(f"An error occurred: {e}")

the following is a conversation with demus hasabis ceo and co-founder of deepmind a company that has published and builds some of the most incredible artificial intelligence systems in the history of computing including alfred zero that learned all by itself to play the game of gold better than any human in the world and alpha fold two that solved protein folding both tasks considered nearly impossible for a very long time demus is widely considered to be one of the most brilliant and impactful humans in the history of artificial intelligence and science and engineering in general this was truly an honor and a pleasure for me to finally sit down with him for this conversation and i'm sure we will talk many times again in the future this is the lex friedman podcast to support it please check out our sponsors in the description and now dear friends here's demis hassabis let's start with a bit of a personal question am i an ai program you wrote to interview people until i get good enough 

In [24]:
transcript_list

[{'text': 'the following is a conversation with',
  'start': 0.08,
  'duration': 3.44},
 {'text': 'demus hasabis', 'start': 1.76, 'duration': 4.96},
 {'text': 'ceo and co-founder of deepmind', 'start': 3.52, 'duration': 5.119},
 {'text': 'a company that has published and builds',
  'start': 6.72,
  'duration': 4.48},
 {'text': 'some of the most incredible artificial',
  'start': 8.639,
  'duration': 4.561},
 {'text': 'intelligence systems in the history of',
  'start': 11.2,
  'duration': 4.8},
 {'text': 'computing including alfred zero that',
  'start': 13.2,
  'duration': 3.68},
 {'text': 'learned', 'start': 16.0, 'duration': 2.96},
 {'text': 'all by itself to play the game of gold',
  'start': 16.88,
  'duration': 4.559},
 {'text': 'better than any human in the world and',
  'start': 18.96,
  'duration': 5.6},
 {'text': 'alpha fold two that solved protein',
  'start': 21.439,
  'duration': 4.241},
 {'text': 'folding', 'start': 24.56, 'duration': 4.16},
 {'text': 'both tasks consider

In [25]:
transcript

"the following is a conversation with demus hasabis ceo and co-founder of deepmind a company that has published and builds some of the most incredible artificial intelligence systems in the history of computing including alfred zero that learned all by itself to play the game of gold better than any human in the world and alpha fold two that solved protein folding both tasks considered nearly impossible for a very long time demus is widely considered to be one of the most brilliant and impactful humans in the history of artificial intelligence and science and engineering in general this was truly an honor and a pleasure for me to finally sit down with him for this conversation and i'm sure we will talk many times again in the future this is the lex friedman podcast to support it please check out our sponsors in the description and now dear friends here's demis hassabis let's start with a bit of a personal question am i an ai program you wrote to interview people until i get good enough

**Text Splitting**

In [26]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])

In [27]:
chunks

[Document(metadata={}, page_content="the following is a conversation with demus hasabis ceo and co-founder of deepmind a company that has published and builds some of the most incredible artificial intelligence systems in the history of computing including alfred zero that learned all by itself to play the game of gold better than any human in the world and alpha fold two that solved protein folding both tasks considered nearly impossible for a very long time demus is widely considered to be one of the most brilliant and impactful humans in the history of artificial intelligence and science and engineering in general this was truly an honor and a pleasure for me to finally sit down with him for this conversation and i'm sure we will talk many times again in the future this is the lex friedman podcast to support it please check out our sponsors in the description and now dear friends here's demis hassabis let's start with a bit of a personal question am i an ai program you wrote to inte

In [28]:
len(chunks)

168

In [29]:
chunks[100]

Document(metadata={}, page_content="and and kind of come up with descriptions of the electron clouds where they're gonna go how they're gonna interact when you put two elements together uh and what we try to do is learn a simulation uh uh learner functional that will describe more chemistry types of chemistry so um until now you know you can run expensive simulations but then you can only simulate very small uh molecules very simple molecules we would like to simulate large materials um and so uh today there's no way of doing that and we're building up towards uh building functionals that approximate schrodinger's equation and then allow you to describe uh what the electrons are doing and all materials sort of science and material properties are governed by the electrons and and how they interact so have a good summarization of the simulation through the functional um but one that is still close to what the actual simulation would come out with so what um how difficult is that to ask w

**Embedding Generation and Store it into Vector Store**

In [33]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

/tmp/ipython-input-3155790237.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [34]:
vector_store = FAISS.from_documents(chunks,embedding)

In [35]:
vector_store.index_to_docstore_id

{0: '07b5a450-5677-41a5-917a-91ee1a87c679',
 1: '6506b8fa-36de-4db4-9d31-c23685428a3c',
 2: 'd81e588d-4396-4ea6-8dc7-966c6d7eead2',
 3: '8a2fefea-03a8-4867-ab34-540913712088',
 4: '137e4b71-d1f3-4552-b3df-8250aaad250a',
 5: '7bd7aaee-41ae-4a22-aa78-a192b07e4da2',
 6: '9c4a8e5a-818d-4c84-a85b-85be7bc94af5',
 7: '064d70ea-1c24-48d2-be94-77cf0528b018',
 8: '6817d2fa-7064-46f2-bf9a-68c4c5313a80',
 9: '5e713400-3cf9-4a89-8e7f-891294d7da94',
 10: '4dc663d4-2c54-4d19-93a0-2ca739ce3aa9',
 11: 'eb3a2ad7-9e58-42fc-9483-4e4831e430eb',
 12: 'a19cc22f-ebb3-4484-95af-8ff150e37595',
 13: '82d6faea-8a9f-4dea-9fd7-721521f9458d',
 14: '31d62424-c6c0-448f-a698-6c7b69b7718a',
 15: 'f469020d-3bc8-4666-83ff-70abb85ed6ef',
 16: '56280999-084a-419b-956d-ae825c528d7f',
 17: '75acc94c-0037-4635-b0ce-cd485d377584',
 18: 'c1fc9936-e3df-4a1f-83dd-ab90d01172f9',
 19: '76267b2b-7c35-4ba4-8d66-87923a667337',
 20: 'e5da42e2-83bf-4c84-8d39-cae451e6888e',
 21: '7b39458e-7314-4eac-9d04-a17631c47962',
 22: '8a4f9428-3342-

In [37]:
vector_store.get_by_ids(['9b7f0c7f-cc5c-435e-b56c-0261da9a0e80'])

[Document(id='9b7f0c7f-cc5c-435e-b56c-0261da9a0e80', metadata={}, page_content='demas establish to support this podcast please check out our sponsors in the description and now let me leave you with some words from edskar dykstra computer science is no more about computers than astronomy is about telescopes thank you for listening and hope to see you next time')]

**Retrieval**

In [38]:
retriever = vector_store.as_retriever(search_type='similarity',search_kwargs={"k":4})

In [39]:
retriever.invoke('what is deepmind')

[Document(id='cebb4949-6081-4232-a52c-456c8cd3dd89', metadata={}, page_content="and how it works this is tough to uh ask you this question because you probably will say it's everything but let's let's try let's try to think to this because you're in a very interesting position where deepmind is the place of some of the most uh brilliant ideas in the history of ai but it's also a place of brilliant engineering so how much of solving intelligence this big goal for deepmind how much of it is science how much is engineering so how much is the algorithms how much is the data how much is the hardware compute infrastructure how much is it the software computer infrastructure yeah um what else is there how much is the human infrastructure and like just the humans interact in certain kinds of ways in all the space of all those ideas how much does maybe like philosophy how much what's the key if um uh if if you were to sort of look back like if we go forward 200 years look back what was the key 

**Augmentation**

In [41]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [42]:
question          = "is the topic of nuclear fusion discussed in this video? if yes then what was discussed"
retrieved_docs    = retriever.invoke(question)

In [43]:
retrieved_docs

[Document(id='96885d36-a3f3-406d-ad96-1b96aebd2428', metadata={}, page_content="in this case in fusion we we collaborated with epfl in switzerland the swiss technical institute who are amazing they have a test reactor that they were willing to let us use which you know i double checked with the team we were going to use carefully and safely i was impressed they managed to persuade them to let us use it and um and it's a it's an amazing test reactor they have there and they try all sorts of pretty crazy experiments on it and um the the the what we tend to look at is if we go into a new domain like fusion what are all the bottleneck problems uh like thinking from first principles you know what are all the bottleneck problems that are still stopping fusion working today and then we look at we you know we get a fusion expert to tell us and then we look at those bottlenecks and we look at the ones which ones are amenable to our ai methods today yes right and and and then and would be intere

In [44]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

"in this case in fusion we we collaborated with epfl in switzerland the swiss technical institute who are amazing they have a test reactor that they were willing to let us use which you know i double checked with the team we were going to use carefully and safely i was impressed they managed to persuade them to let us use it and um and it's a it's an amazing test reactor they have there and they try all sorts of pretty crazy experiments on it and um the the the what we tend to look at is if we go into a new domain like fusion what are all the bottleneck problems uh like thinking from first principles you know what are all the bottleneck problems that are still stopping fusion working today and then we look at we you know we get a fusion expert to tell us and then we look at those bottlenecks and we look at the ones which ones are amenable to our ai methods today yes right and and and then and would be interesting from a research perspective from our point of view from an ai point of\n\

In [45]:
final_prompt = prompt.invoke({'context':context_text, 'question':question})

**Generations**

In [46]:
model = ChatGoogleGenerativeAI(model='gemini-2.5-flash')

In [47]:
result = model.invoke(final_prompt)

In [48]:
print(result.content)

Yes, the topic of nuclear fusion is discussed.

What was discussed:
*   Nuclear fusion is identified as an area where AI can provide significant help, especially for energy and climate challenges.
*   The company collaborated with EPFL (Swiss Technical Institute) in Switzerland, using their test reactor.
*   They focus on identifying bottleneck problems in fusion that can be addressed using AI methods.
*   They published a Nature paper on using deep reinforcement learning (deep RL) for the magnetic control of tokamak plasmas.
*   This work involved holding and controlling the plasma in specific shapes for a record amount of time, essentially "carving" the plasma.
*   They are currently talking to fusion startups to identify the next problems to tackle in the field.
*   It was noted that fusion has many challenges, primarily in physics, material science, and engineering.


**Using Chains**

In [49]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [50]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [51]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [53]:
parser = StrOutputParser()
final_chain = parallel_chain | prompt | model | parser

In [54]:
final_chain.invoke('summarize the video')

"The transcript discusses the need for a deeper, simpler explanation of physics beyond the current standard model, which is seen as incomplete. This new understanding could offer insights into long-standing mysteries such as consciousness, life, and gravity. It also touches on the nature of intelligence, suggesting that a key sign of intelligence is the ability to explain complex topics simply, referencing Richard Feynman. Finally, the conversation explores whether solving intelligence, particularly at a place like DeepMind, is primarily a matter of scientific ideas and algorithms or engineering aspects like data, hardware, software, and human infrastructure, concluding that it's a combination of these elements."